In [3]:
MONGO_URI = "mongodb://10.3.32.15:27017/"
DB_NAME = "main"
COLLECTION_NAME = "tracking_sesssions_v3_b"

In [4]:
from pymongo import MongoClient

# --- Placeholders ---


def deduplicate_uuids():
    # Connect to MongoDB
    client = MongoClient(MONGO_URI)
    db = client[DB_NAME]
    collection = db[COLLECTION_NAME]

    print("Scanning collection for duplicate uuids...")
    
    # Pipeline to find all uuids that appear more than once
    pipeline = [
        {
            "$group": {
                "_id": "$uuid",
                # Store all _ids associated with this uuid
                "dups": {"$push": "$_id"},
                # Count how many times the uuid appears
                "count": {"$sum": 1}
            }
        },
        {
            # Filter to only pass through the duplicates
            "$match": {
                "count": {"$gt": 1}
            }
        }
    ]

    # allowDiskUse=True prevents memory errors on massive collections
    cursor = collection.aggregate(pipeline, allowDiskUse=True)

    ids_to_delete = []
    
    # Process the results
    for doc in cursor:
        # Keep the first _id (skip index 0), add the rest to our deletion list
        duplicates = doc["dups"][1:] 
        ids_to_delete.extend(duplicates)

    if not ids_to_delete:
        print("No duplicate uuids found. The collection is clean!")
        return

    print(f"Found {len(ids_to_delete)} duplicate documents. Starting deletion...")

    # Delete the duplicates in chunks to avoid the 16MB BSON size limit 
    # when using the $in operator with massive lists.
    chunk_size = 10000
    total_deleted = 0
    
    for i in range(0, len(ids_to_delete), chunk_size):
        chunk = ids_to_delete[i:i + chunk_size]
        result = collection.delete_many({"_id": {"$in": chunk}})
        total_deleted += result.deleted_count
        print(f"  -> Deleted {total_deleted} / {len(ids_to_delete)} duplicates...")

    print("Deduplication complete!")

if __name__ == "__main__":
    deduplicate_uuids()

Scanning collection for duplicate uuids...
No duplicate uuids found. The collection is clean!
